# 🏛️ NEX-CIV: 10,000 Real Multilingual Civic Grievance Classifier
### Epoch-Based Training on Real Datasets for Department (5 Classes) & Complaint Type (20 Classes)
**Trained across Epochs on 10,000 Records: Pure Telugu (`te`), English (`en`), and Romanized Telugu (`te_latn`)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

---
### 📌 Research Specifications:
1. **Real Datasets (10,000 Samples)**:
   - **BBMP Citizen Grievances**: Sourced from Bengaluru Smart City / BBMP open data (`citizen-grievances.csv.gz`).
   - **Andhra Pradesh Municipal Records**: VMC (Vijayawada), GVMC (Vizag), GMC (Guntur), TMC (Tirupati), Eluru ULBs.
   - **CivicDex NLP Reference**: Multilingual public request mapping.
2. **Language Balance (10,000 Records)**:
   - Pure Telugu (`te`): ~3,334 records
   - Pure English (`en`): ~3,333 records
   - Romanized Telugu (`te_latn`): ~3,333 records
3. **Two-Task System**:
   - **Task 1: Department Classification** (5 classes: Roads, Drainage, Water Supply, Electrical, Sanitation)
   - **Task 2: Complaint Type Classification** (20 specific municipal failure modes: Potholes, Overflow, Contamination, etc.)
4. **Training Across Epochs**:
   - Multi-task training loop across **5-10 Epochs** with batch processing.
   - Combined Multi-Task Loss: $\mathcal{L} = \mathcal{L}_{\text{department}} + \mathcal{L}_{\text{complaint\_type}}$
   - Loss and Accuracy tracked per epoch with Matplotlib learning curves.
5. **Disaggregated Testing (1,500 Held-Out Samples)**:
   - Evaluates whether NexCiv independently understands Telugu, English, and Romanized Telugu.

In [ ]:
# Step 1: Install & Setup Environment
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib

import os
import re
import random
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix, log_loss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("✅ Libraries loaded and random seed initialized.")

## 2. Ingestion of 10,000 Real Multilingual Civic Complaints
Loads `nexciv_multilingual_complaints_10k.csv` directly from multiple integrated real sources (BBMP Citizen Grievances open data, AP Municipal Records, and CivicDex).

In [ ]:
CSV_PATH = 'nexciv_multilingual_complaints_10k.csv'

# If running in Google Colab fresh session, download the master 10,000 dataset
if not os.path.exists(CSV_PATH):
    print("Downloading 10,000 Real Multilingual Civic Dataset...")
    import urllib.request
    repo_url = 'https://raw.githubusercontent.com/pranaya-s/NEXCIV/main/nexciv_multilingual_complaints_10k.csv'
    try:
        urllib.request.urlretrieve(repo_url, CSV_PATH)
        print("Successfully downloaded from GitHub.")
    except Exception as e:
        print("Could not fetch remote, checking local fallback...")
        # Fallback to local files if present
        for p in ['ai/datasets/processed/nexciv_multilingual_complaints_10k.csv', 'nexciv_multilingual_complaints_v1.csv']:
            if os.path.exists(p):
                CSV_PATH = p
                break

df = pd.read_csv(CSV_PATH)
print(f"📂 Dataset Successfully Loaded: {len(df)} records.")
display(df.head(6))

print("\nLanguage Style Distribution (Target ~3,333 each):")
display(df['language_style'].value_counts().to_frame('Record Count'))

print("\nDepartment Distribution (5 Classes):")
display(df['department'].value_counts().to_frame('Record Count'))

print("\nComplaint Type Distribution (20 Failure Modes):")
display(df['complaint_type'].value_counts().to_frame('Record Count'))

## 3. Dataset Splitting: 70% Train (7,000), 15% Val (1,500), 15% Test (1,500)
We use the pre-stratified splits ensuring balanced representation of all 5 departments and 20 complaint types.

In [ ]:
train_df = df[df['split'] == 'train'].copy()
val_df = df[df['split'] == 'validation'].copy()
test_df = df[df['split'] == 'test'].copy()

print(f"✅ Training Set:   {len(train_df)} records (70%)")
print(f"✅ Validation Set: {len(val_df)} records (15%)")
print(f"✅ Test Set:       {len(test_df)} records (15%)")

## 4. NLP Preprocessing & Sublinear TF-IDF Vectorization
Normalizes Unicode Telugu script (NFC) and extracts sublinear n-gram feature representations.

In [ ]:
tfidf = TfidfVectorizer(
    token_pattern=r'(?u)\b\w+\b',
    ngram_range=(1, 2),
    max_features=15000,
    sublinear_tf=True
)

X_train = tfidf.fit_transform(train_df['complaint_text'])
X_val = tfidf.transform(val_df['complaint_text'])
X_test = tfidf.transform(test_df['complaint_text'])

y_dept_train = train_df['department'].values
y_dept_val = val_df['department'].values
y_dept_test = test_df['department'].values

y_type_train = train_df['complaint_type'].values
y_type_val = val_df['complaint_type'].values
y_type_test = test_df['complaint_type'].values

dept_classes = np.unique(y_dept_train)
type_classes = np.unique(y_type_train)

print(f"TF-IDF Vocabulary Size: {len(tfidf.vocabulary_)} n-grams")
print(f"Department Classes ({len(dept_classes)}): {list(dept_classes)}")
print(f"Complaint Type Classes ({len(type_classes)}): {list(type_classes)}")

## 5. Multi-Task Training Loop Across EPOCHS
Trains the dual-task models across **Epochs** with mini-batch incremental updates:
- **Loss Computation**: Cross-Entropy Loss computed for both Department and Complaint Type
- **Combined Multi-Task Loss**: $\mathcal{L} = \mathcal{L}_{\text{department}} + \mathcal{L}_{\text{complaint\_type}}$
- **Epoch-by-Epoch Progress**: Evaluates Validation Loss and Validation Accuracy after every epoch.

In [ ]:
EPOCHS = 5
BATCH_SIZE = 256

# Incremental Multi-Task Classifiers using Log-Loss (Cross-Entropy)
dept_model = SGDClassifier(loss='log_loss', penalty='l2', alpha=1e-4, random_state=SEED)
type_model = SGDClassifier(loss='log_loss', penalty='l2', alpha=1e-4, random_state=SEED)

history = {
    'epoch': [],
    'train_loss': [],
    'val_loss': [],
    'val_dept_acc': [],
    'val_type_acc': []
}

n_samples = X_train.shape[0]
n_batches = int(np.ceil(n_samples / BATCH_SIZE))

print("=" * 75)
print(f"STARTING MULTI-TASK EPOCH TRAINING (EPOCHS={EPOCHS}, BATCHES_PER_EPOCH={n_batches})")
print("=" * 75)

for epoch in range(1, EPOCHS + 1):
    # Shuffle training indices per epoch
    perm = np.random.permutation(n_samples)
    X_train_shuffled = X_train[perm]
    y_dept_shuffled = y_dept_train[perm]
    y_type_shuffled = y_type_train[perm]
    
    # Mini-batch gradient descent
    for b in range(n_batches):
        s_idx = b * BATCH_SIZE
        e_idx = min(s_idx + BATCH_SIZE, n_samples)
        X_batch = X_train_shuffled[s_idx:e_idx]
        y_d_b = y_dept_shuffled[s_idx:e_idx]
        y_t_b = y_type_shuffled[s_idx:e_idx]
        
        if epoch == 1 and b == 0:
            dept_model.partial_fit(X_batch, y_d_b, classes=dept_classes)
            type_model.partial_fit(X_batch, y_t_b, classes=type_classes)
        else:
            dept_model.partial_fit(X_batch, y_d_b)
            type_model.partial_fit(X_batch, y_t_b)
            
    # Compute Cross-Entropy Losses
    d_tr_p = dept_model.predict_proba(X_train)
    t_tr_p = type_model.predict_proba(X_train)
    d_vl_p = dept_model.predict_proba(X_val)
    t_vl_p = type_model.predict_proba(X_val)
    
    train_loss = log_loss(y_dept_train, d_tr_p, labels=dept_classes) + log_loss(y_type_train, t_tr_p, labels=type_classes)
    val_loss = log_loss(y_dept_val, d_vl_p, labels=dept_classes) + log_loss(y_type_val, t_vl_p, labels=type_classes)
    
    v_d_acc = accuracy_score(y_dept_val, dept_model.predict(X_val))
    v_t_acc = accuracy_score(y_type_val, type_model.predict(X_val))
    
    history['epoch'].append(epoch)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_dept_acc'].append(v_d_acc)
    history['val_type_acc'].append(v_t_acc)
    
    print(f"Epoch {epoch:2d}/{EPOCHS:2d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Dept Acc: {v_d_acc*100:5.2f}% | Val Type Acc: {v_t_acc*100:5.2f}%")

print("=" * 75)
print("✅ EPOCH TRAINING COMPLETE!")
print("=" * 75)

## 6. Learning Curves: Loss & Accuracy Across Epochs

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Loss vs Epoch
ax1.plot(history['epoch'], history['train_loss'], 'b-o', label='Training Loss', linewidth=2)
ax1.plot(history['epoch'], history['val_loss'], 'r--s', label='Validation Loss', linewidth=2)
ax1.set_title('Multi-Task Cross-Entropy Loss per Epoch', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=11)
ax1.set_ylabel('Combined Multi-Task Loss', fontsize=11)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend()

# Subplot 2: Accuracy vs Epoch
ax2.plot(history['epoch'], [acc*100 for acc in history['val_dept_acc']], 'g-o', label='Task 1: Department Acc', linewidth=2)
ax2.plot(history['epoch'], [acc*100 for acc in history['val_type_acc']], 'm--^', label='Task 2: Complaint Type Acc', linewidth=2)
ax2.set_title('Validation Accuracy Progression across Epochs', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=11)
ax2.set_ylabel('Validation Accuracy (%)', fontsize=11)
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.show()

## 7. Critical Experiment: Disaggregated Per-Language Evaluation (1,500 Test Set)
Evaluates whether the model demonstrates genuine comprehension on:
1. **Pure Telugu (`te`)**
2. **Pure English (`en`)**
3. **Romanized Telugu (`te_latn`)**

In [ ]:
test_d_preds = dept_model.predict(X_test)
test_t_preds = type_model.predict(X_test)

print("=" * 75)
print("🎯 FINAL HELD-OUT TEST PERFORMANCE (1,500 SAMPLES)")
print("=" * 75)
print(f"Task 1: Department Accuracy:     {accuracy_score(y_dept_test, test_d_preds)*100:.2f}%")
print(f"Task 2: Complaint Type Accuracy: {accuracy_score(y_type_test, test_t_preds)*100:.2f}%")

print("\n" + "=" * 75)
print("🔬 DISAGGREGATED PERFORMANCE BY WRITING STYLE (1,500 TEST SAMPLES)")
print("=" * 75)

lang_rows = []
for lang in ['te', 'en', 'te_latn']:
    mask = (test_df['language_style'] == lang).values
    sub_d_preds = dept_model.predict(X_test[mask])
    sub_t_preds = type_model.predict(X_test[mask])
    
    d_acc = accuracy_score(y_dept_test[mask], sub_d_preds)
    d_f1 = f1_score(y_dept_test[mask], sub_d_preds, average='macro')
    t_acc = accuracy_score(y_type_test[mask], sub_t_preds)
    t_f1 = f1_score(y_type_test[mask], sub_t_preds, average='macro')
    
    lang_name = 'Pure Telugu (తెలుగు)' if lang=='te' else ('Pure English' if lang=='en' else 'Romanized Telugu (Tenglish)')
    lang_rows.append({
        'Language Style': lang_name,
        'Test Samples': np.sum(mask),
        'Dept Accuracy': f"{d_acc*100:.2f}%",
        'Dept Macro F1': f"{d_f1:.4f}",
        'Complaint Type Acc': f"{t_acc*100:.2f}%",
        'Complaint Type Macro F1': f"{t_f1:.4f}"
    })

display(pd.DataFrame(lang_rows))

# Overall Department Classification Report
print("\nOVERALL DEPARTMENT CLASSIFICATION REPORT (1,500 SAMPLES):")
print(classification_report(y_dept_test, test_d_preds, digits=4))

## 8. Department Confusion Matrix Heatmap

In [ ]:
cm = confusion_matrix(y_dept_test, test_d_preds, labels=dept_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=dept_classes, yticklabels=dept_classes)
plt.title(f'NEX-CIV Department Confusion Matrix (1,500 Test Samples)', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Department', fontsize=11)
plt.ylabel('True Ground Truth Department', fontsize=11)
plt.tight_layout()
plt.show()

## 9. Export Production Drop-in `.pkl` Bundle for NEX-CIV
Exports standard drop-in pipelines compatible with `backend/classifier.py` and the citizen/department dashboards.

In [ ]:
bundle = {
    'wing_pipeline': Pipeline([('tfidf', tfidf), ('clf', dept_model)]),
    'pipeline': Pipeline([('tfidf', tfidf), ('clf', dept_model)]),
    'complaint_type_pipeline': Pipeline([('tfidf', tfidf), ('clf', type_model)]),
    'classes': dept_classes,
    'complaint_types': type_classes,
    'model_version': '10k-epochs-multitask',
    'sample_count': 10000,
    'epochs_trained': EPOCHS
}

out_path = 'complaint_classifier.pkl'
joblib.dump(bundle, out_path)
joblib.dump(bundle, 'nexciv_multimodal_priority_model.pkl')

print(f"✅ Successfully exported 10k epoch-trained bundle: '{out_path}' ({os.path.getsize(out_path)/1024:.1f} KB)")
print("\n👉 Run in Google Colab to download directly to your PC:")
print("   from google.colab import files")
print("   files.download('complaint_classifier.pkl')")

# from google.colab import files
# files.download(out_path)